# 03_features — 도메인 근거 기반 피처 엔지니어링

**목적(Why)**: `02_eda.ipynb`/`reports/02_eda.md`에서 확인한 사실들을 실제로 학습에 쓸 피처로 만듭니다. 이 노트북에서 가장 중요한 결정은 3절(풍속 추정)입니다 — 처음엔 물리 공식(윈드시어 멱법칙)으로 허브고도 풍속을 만들려고 했는데 실제로 해보니 오히려 안 좋아져서, SCADA를 정답 삼아 여러 예보 풍속을 회귀로 결합하는 방식으로 바꿨습니다. **이 판단이 틀렸을 가능성까지 감안해서, 시도 과정과 검증 수치를 전부 셀로 남겨둡니다** — 나중에 다시 보고 틀렸다고 판단되면 3절만 바꾸면 됩니다.

**입력**: `data/processed/train_base.parquet`, `data/processed/test_base.parquet` (`01_preprocessing.ipynb` 산출물), `data/train/scada_vestas_train.csv`/`scada_unison_train.csv` (3절 회귀 학습 전용, train에만 존재)
**출력**: `data/processed/train_features_v1.parquet`, `data/processed/test_features_v1.parquet`

**체크리스트 진행 상황**: 1절(결측·이상구간 처리)~7절(시간 피처) 이번 배치. 8절(GFS-LDAPS 앙상블 차이), 9절(결빙 위험 피처)은 다음 배치.

## 0. 환경 설정

라이브러리·경로·상수를 정의하고 `01_preprocessing.ipynb`가 만든 캐시를 불러옵니다.

In [1]:
import sys
print(sys.executable)  # venv 안의 파이썬을 쓰고 있는지 확인

d:\공모전\wind_forecast_new\venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

SEED = 42
np.random.seed(SEED)

PROCESSED_DIR = "../data/processed"

CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}
GROUP_COLS = list(CAPACITY_KWH.keys())
HUB_HEIGHT_M = 117  # info.xlsx 확인값, 3개 그룹 모두 동일

train_base = pd.read_parquet(f"{PROCESSED_DIR}/train_base.parquet")
test_base = pd.read_parquet(f"{PROCESSED_DIR}/test_base.parquet")
print("train_base:", train_base.shape, "| test_base:", test_base.shape)

train_base: (26304, 804) | test_base: (8760, 798)


**확인할 것**: `train_base`가 `(26304, 804)`, `test_base`가 `(8760, 798)`인지 — 01_preprocessing/02_eda에서 본 값과 같아야 합니다.

## 1. 결측·이상구간 처리 (피처 만들기 전에 먼저)

피처를 만들기 시작한 뒤에 결측 방침을 바꾸면 어디까지 다시 계산해야 하는지 헷갈립니다. 그래서 `leakage-guard`·`preprocessing` 스킬과 `reports/02_eda.md`에서 확정한 방침을 가장 먼저 적용합니다.

- 1-1: `kpx_group_1/2`의 라벨(발전량) 결측 처리
- 1-2: test의 LDAPS 예보 결측(발표 누락 추정 3개 시점) 처리
- 1-3: curtailment(출력제한/정지) 확정 구간을 학습 라벨에서 제외

### 1-1. `kpx_group_1/2` 라벨 결측 처리

**무엇을**: `reports/02_eda.md` 2절에서 확인한 대로, group_1/2에는 2022-10-24~27 나흘 연속 결측과 그 외 산발적(1~11시간) 결측이 섞여 있습니다.

**왜 이렇게 나누는가**: 나흘(최대 96시간) 연속 결측은 그 사이 풍향·전선 통과 등으로 발전량이 어떻게 움직였을지 짐작할 근거가 너무 약해서, **보간하지 않고 그대로 결측으로 둡니다**(나중에 그룹별 학습 데이터를 만들 때 `dropna`로 자연히 빠집니다 — 별도로 행을 지우는 코드가 필요 없습니다). 반면 6시간 이내의 짧은 결측은 바람이 그 사이 급변했을 가능성이 낮아 선형보간이 합리적입니다(풍력의 시간 단위 지속성 가정, 도메인 근거).

**leakage 관점**: 이건 **라벨(정답) 보간**이지 예측 시점에 쓰는 입력 피처가 아닙니다. leakage-guard가 경고하는 "미래 값을 쓰는 보간 금지"는 test에서 재현 불가능한 입력 피처에 대한 이야기이고, 여기서는 이미 지나간 train 기간의 정답값을 무엇으로 채울지 정하는 것뿐이라 문제가 없습니다.

In [3]:
LONG_GAP_START = pd.Timestamp("2022-10-24")
LONG_GAP_END = pd.Timestamp("2022-10-28")  # 02_eda 2절에서 확인한 4일 연속 결측 구간(끝 여유 포함)

for col in ["kpx_group_1", "kpx_group_2"]:
    in_long_gap = train_base["kst_dtm"].between(LONG_GAP_START, LONG_GAP_END) & train_base[col].isna()
    print(col, "긴 결측 구간(2022-10-24~27) 내 결측:", in_long_gap.sum(), "건 -> 보간하지 않고 그대로 둠")

for col in ["kpx_group_1", "kpx_group_2"]:
    before = train_base[col].isna().sum()
    # limit=6: 6시간 이내 짧은 결측만 선형보간. 4일짜리 긴 구간은 limit을 넘어서므로 채워지지 않음
    train_base[col] = train_base[col].interpolate(method="linear", limit=6, limit_area="inside")
    after = train_base[col].isna().sum()
    print(col, f"보간 전 결측 {before}건 -> 보간 후 {after}건")

kpx_group_1 긴 결측 구간(2022-10-24~27) 내 결측: 82 건 -> 보간하지 않고 그대로 둠
kpx_group_2 긴 결측 구간(2022-10-24~27) 내 결측: 82 건 -> 보간하지 않고 그대로 둠
kpx_group_1 보간 전 결측 104건 -> 보간 후 81건
kpx_group_2 보간 전 결측 103건 -> 보간 후 81건


**확인할 것**: 보간 후 결측이 81건 근처로 줄어드는지(104건 중 짧은 결측 20여 건만 채워지고, 나흘 구간은 그대로 남아야 합니다).

### 1-2. test LDAPS 예보 결측 처리

**무엇을/왜**: `reports/02_eda.md` 2절에서 확인한 test_base의 결측 752건은 2025-04-08 17시·06-18 18시·07-18 06시 **3개 시점에만** 몰려 있었습니다(예보 발표 누락 추정). 시간순 정렬 후 **직전 값으로 채웁니다**(`ffill`) — 미래 시점 값을 쓰지 않으므로 leakage가 없습니다.

In [4]:
cols_with_na = test_base.columns[test_base.isna().any()].tolist()
print("결측 있는 컬럼 수(처리 전):", len(cols_with_na))

test_base = test_base.sort_values("kst_dtm").reset_index(drop=True)
test_base[cols_with_na] = test_base[cols_with_na].ffill()

print("결측 총량(처리 후):", test_base.isna().sum().sum())

결측 있는 컬럼 수(처리 전): 304
결측 총량(처리 후): 0


**확인할 것**: 처리 후 결측 총량이 0인지.

### 1-3. curtailment(출력제한/정지) 확정 구간을 학습 라벨에서 제외

**무엇을**: `reports/02_eda.md` 4·6절, `HANDOFF.md`에서 확정한 3개 구간 — SCADA 풍속이 충분(4~8 m/s)한데도 발전량이 0이었고, 컷아웃(순간 최고풍속 20.4 m/s로 25 m/s 미만)도 아니라서 계통 제약·정비로 판단한 구간입니다. 3개 그룹 모두 같은 시기에 동시 발생했습니다.

| 시작 | 끝(여유 포함) |
|---|---|
| 2023-02-13 | 2023-02-17 |
| 2024-01-18 | 2024-01-24 |
| 2024-02-22 | 2024-03-01 |

**왜 삭제하는가**: 이 구간을 그대로 두면 모델이 "바람이 세도 가끔 발전량이 0"이라는, 풍속-발전량의 진짜 물리적 관계와 무관한 패턴을 학습하게 됩니다.

**왜 안전한가(중요)**: 대회 평가 산식(`src/metric.py`)은 **실제 발전량이 설비용량의 10% 이상인 시간대만** 채점합니다. curtailment 구간은 발전량이 0이라 애초에 채점 대상이 아닙니다. 그래서 test(2025년)에 비슷한 정지가 또 있어도 **그 시간을 못 맞히는 건 점수에 영향이 없습니다.** 즉 이 처리는 "test에서 못 만드는 정보를 몰래 쓰는 것"이 아니라 그냥 "학습에 안 쓰는 것"뿐이고, test 쪽에는 아무 것도 적용하지 않으므로 leakage 문제가 없습니다.

In [5]:
CURTAILMENT_WINDOWS = [
    ("2023-02-13", "2023-02-17"),
    ("2024-01-18", "2024-01-24"),
    ("2024-02-22", "2024-03-01"),
]  # reports/02_eda.md 4·6절 확정 근거, HANDOFF.md 결정사항 참고

for start, end in CURTAILMENT_WINDOWS:
    mask = train_base["kst_dtm"].between(pd.Timestamp(start), pd.Timestamp(end))
    for col in GROUP_COLS:
        n = train_base.loc[mask, col].notna().sum()
        train_base.loc[mask, col] = np.nan
        print(f"{col}: {start}~{end} 구간 {n}건을 학습 라벨에서 제외(NaN 처리)")

kpx_group_1: 2023-02-13~2023-02-17 구간 97건을 학습 라벨에서 제외(NaN 처리)
kpx_group_2: 2023-02-13~2023-02-17 구간 97건을 학습 라벨에서 제외(NaN 처리)
kpx_group_3: 2023-02-13~2023-02-17 구간 97건을 학습 라벨에서 제외(NaN 처리)
kpx_group_1: 2024-01-18~2024-01-24 구간 145건을 학습 라벨에서 제외(NaN 처리)
kpx_group_2: 2024-01-18~2024-01-24 구간 145건을 학습 라벨에서 제외(NaN 처리)
kpx_group_3: 2024-01-18~2024-01-24 구간 145건을 학습 라벨에서 제외(NaN 처리)
kpx_group_1: 2024-02-22~2024-03-01 구간 193건을 학습 라벨에서 제외(NaN 처리)
kpx_group_2: 2024-02-22~2024-03-01 구간 193건을 학습 라벨에서 제외(NaN 처리)
kpx_group_3: 2024-02-22~2024-03-01 구간 193건을 학습 라벨에서 제외(NaN 처리)


**확인할 것**: 그룹당 3개 구간 합쳐서 400건 안팎이 제외되는지. (이 처리는 라벨에만 적용되고 GFS/LDAPS 입력 컬럼은 그대로 남아있습니다 — test에 같은 시기 정지가 있어도 입력 자체는 정상적으로 예측에 쓸 수 있어야 하기 때문입니다.)

## 2. 그룹별 풍속 원료 계산 (회귀 피처의 재료)

**무엇을/왜**: 3절에서 만들 회귀 피처의 입력 재료로, 그룹별 최근접 LDAPS 격자 10m 풍속·풍향, LDAPS 16격자 평균 풍속, GFS(전 그룹 공통 g5) 높이별(10/80/100m/850hPa) 풍속을 계산합니다. `wind_speed()`/`wind_direction_deg()`는 u(동서 성분)/v(남북 성분)로부터 풍속·풍향을 구하는 함수로, `02_eda.ipynb`와 동일합니다. **train/test에 완전히 같은 함수·같은 로직을 적용합니다**(leakage-guard 원칙 — 전처리는 항상 두 데이터에 동일하게).

In [6]:
def wind_speed(df, u_col, v_col):
    return np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)


def wind_direction_deg(df, u_col, v_col):
    """기상학적 풍향(바람이 '불어오는' 방향, 0=북 90=동 180=남 270=서, 도 단위)."""
    return (270 - np.degrees(np.arctan2(df[v_col], df[u_col]))) % 360


GROUP_NEAREST_LDAPS = {"kpx_group_1": 5, "kpx_group_2": 6, "kpx_group_3": 12}  # 02_eda 2절에서 확정
GFS_NEAREST_GRID = 5  # 전 그룹 공통 (02_eda 2절 — 터빈 전체가 GFS 격자 하나 안에 들어감)

GFS_HEIGHT_COLS = {
    "10m": ("heightAboveGround_10_10u", "heightAboveGround_10_10v"),
    "80m": ("heightAboveGround_80_u", "heightAboveGround_80_v"),
    "100m": ("heightAboveGround_100_100u", "heightAboveGround_100_100v"),
    "850hPa": ("isobaricInhPa_850_u", "isobaricInhPa_850_v"),
}

for df in [train_base, test_base]:
    for group_col, grid_id in GROUP_NEAREST_LDAPS.items():
        u_col = f"ldaps_g{grid_id}_heightAboveGround_10_10u"
        v_col = f"ldaps_g{grid_id}_heightAboveGround_10_10v"
        df[f"{group_col}_ws10_nearest"] = wind_speed(df, u_col, v_col)
        df[f"{group_col}_wd"] = wind_direction_deg(df, u_col, v_col)

    ldaps_speeds = [wind_speed(df, f"ldaps_g{g}_heightAboveGround_10_10u", f"ldaps_g{g}_heightAboveGround_10_10v") for g in range(1, 17)]
    df["ldaps_ws10_avg16"] = pd.concat(ldaps_speeds, axis=1).mean(axis=1)

    for height, (us, vs) in GFS_HEIGHT_COLS.items():
        df[f"gfs_g5_ws_{height}"] = wind_speed(df, f"gfs_g{GFS_NEAREST_GRID}_{us}", f"gfs_g{GFS_NEAREST_GRID}_{vs}")

train_base[[f"{g}_ws10_nearest" for g in GROUP_COLS] + ["ldaps_ws10_avg16"] + [f"gfs_g5_ws_{h}" for h in GFS_HEIGHT_COLS]].describe()

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\793265412.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_ws10_nearest"] = wind_speed(df, u_col, v_col)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\793265412.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_wd"] = wind_direction_deg(df, u_col, v_col)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\793265412.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,

,kpx_group_1_ws10_nearest,kpx_group_2_ws10_nearest,kpx_group_3_ws10_nearest,ldaps_ws10_avg16,gfs_g5_ws_10m,gfs_g5_ws_80m,gfs_g5_ws_100m,gfs_g5_ws_850hPa
count,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000,26304.000000
mean,4.779440,5.091271,5.196616,4.836247,2.519790,3.783026,3.967223,8.065915
std,2.434159,2.526631,2.765333,2.479576,1.655472,2.961447,3.165436,6.377717
min,0.037558,0.038639,0.072987,0.371642,0.012522,0.020277,0.001464,0.017597
25%,3.056498,3.180371,2.989968,2.965037,1.456602,1.867905,1.905393,2.817942
50%,4.259118,4.646858,4.691264,4.321871,2.156342,2.914652,3.001930,6.057475
75%,6.017516,6.619826,7.039176,6.278099,3.003918,4.615204,4.886699,12.210562
max,17.220675,17.583720,17.304098,16.874288,15.305853,25.053681,26.482454,39.887524


**확인할 것**: 평균값들이 02_eda에서 본 값과 비슷한지(예: `ldaps_ws10_avg16` 평균 4.84 m/s 근처).

### 2-1. (근거) 최근접 격자 vs 16격자평균 — 그룹마다 정말 다른가

3-1의 회귀에는 최근접 LDAPS(`{group}_ws10_nearest`)와 16격자평균(`ldaps_ws10_avg16`)을 둘 다 입력으로 넣습니다. 그 이유를 코드로 남깁니다 — 대화 중 "group_1만 평균이 더 좋고 group_2/3은 최근접이 더 좋다"는 게 우연 아니냐는 질문이 나와서, **연도별로 나눠서**(2022/2023/2024 각각), 그리고 **통계적 유의성 검정**(Steiger's Z — 같은 대상과 상관인 두 변수 중 어느 쪽이 유의하게 더 높은지 비교하는 검정)으로 확인했습니다.

In [7]:
from scipy import stats


def steiger_z_dependent(r_a, r_b, r_ab, n):
    """종속적인(같은 대상 Y와 상관인) 두 상관계수 r_a, r_b의 차이가 유의한지 검정 (Steiger 1980).
    r_ab: 두 예측 변수 A, B 사이의 상관."""
    rm2 = (r_a ** 2 + r_b ** 2) / 2
    z_a, z_b = np.arctanh(r_a), np.arctanh(r_b)
    cov = (r_ab * (1 - 2 * rm2) - 0.5 * rm2 * (1 - 2 * rm2 - r_ab ** 2)) / (1 - rm2) ** 2
    se = np.sqrt(max((2 - 2 * cov) / (n - 3), 1e-12))
    z = (z_a - z_b) / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p


train_base["year"] = train_base["kst_dtm"].dt.year

print("연도별 상관(최근접 vs 16격자평균):")
for group_col in GROUP_COLS:
    for yr in [2022, 2023, 2024]:
        sub = train_base[train_base["year"] == yr]
        if sub[group_col].notna().sum() < 100:
            continue
        c_near = sub[f"{group_col}_ws10_nearest"].corr(sub[group_col])
        c_avg = sub["ldaps_ws10_avg16"].corr(sub[group_col])
        print(f"  {group_col} {yr}: nearest={c_near:.4f}  avg16={c_avg:.4f}  차이(avg-near)={c_avg-c_near:+.4f}")

print()
print("전체기간 유의성(Steiger's Z, 두 상관계수 차이가 우연일 확률 p):")
for group_col in GROUP_COLS:
    sub = train_base.dropna(subset=[group_col, f"{group_col}_ws10_nearest", "ldaps_ws10_avg16"])
    n = len(sub)
    r_near = sub[f"{group_col}_ws10_nearest"].corr(sub[group_col])
    r_avg = sub["ldaps_ws10_avg16"].corr(sub[group_col])
    r_between = sub[f"{group_col}_ws10_nearest"].corr(sub["ldaps_ws10_avg16"])
    z, p = steiger_z_dependent(r_near, r_avg, r_between, n)
    print(f"  {group_col}: n={n}  r_nearest={r_near:.4f}  r_avg16={r_avg:.4f}  Z={z:.2f}  p={p:.2e}")

연도별 상관(최근접 vs 16격자평균):
  kpx_group_1 2022: nearest=0.6226  avg16=0.7363  차이(avg-near)=+0.1137
  kpx_group_1 2023: nearest=0.6029  avg16=0.7180  차이(avg-near)=+0.1151
  kpx_group_1 2024: nearest=0.6583  avg16=0.7684  차이(avg-near)=+0.1101
  kpx_group_2 2022: nearest=0.7650  avg16=0.7485  차이(avg-near)=-0.0165
  kpx_group_2 2023: nearest=0.7420  avg16=0.7238  차이(avg-near)=-0.0182
  kpx_group_2 2024: nearest=0.7872  avg16=0.7728  차이(avg-near)=-0.0144
  kpx_group_3 2023: nearest=0.7383  avg16=0.6925  차이(avg-near)=-0.0457
  kpx_group_3 2024: nearest=0.8266  avg16=0.7861  차이(avg-near)=-0.0405

전체기간 유의성(Steiger's Z, 두 상관계수 차이가 우연일 확률 p):
  kpx_group_1: n=25788  r_nearest=0.6250  r_avg16=0.7380  Z=-90.51  p=0.00e+00
  kpx_group_2: n=25788  r_nearest=0.7640  r_avg16=0.7476  Z=33.17  p=0.00e+00
  kpx_group_3: n=17103  r_nearest=0.7827  r_avg16=0.7397  Z=39.67  p=0.00e+00


**확인할 것**: 위쪽 표에서 group_1은 3년 내내 avg16이 더 높고(+0.11~0.12), group_2/3은 3년 내내 nearest가 더 높은지(-0.02~-0.05) — 즉 방향이 3년 동안 한 번도 안 바뀌는지. 아래쪽 p값이 전부 매우 작은지(0.001보다 훨씬 작아야 "우연이 아니다"라고 말할 수 있습니다). **이 결과가 3절에서 두 피처를 모두 회귀 입력으로 넣는 근거입니다** — 회귀가 그룹마다 유리한 쪽에 자동으로 더 큰 가중치를 주게 됩니다(3-3의 회귀계수를 보면 실제로 group_1은 avg16 계수가 더 크고 group_2/3은 nearest 계수가 더 큽니다).

## 3. SCADA 회귀 기반 "추정 실제 풍속" 피처 (이 노트북의 핵심)

### 왜 이 방식을 쓰는가 — 시도 기록 (틀렸을 경우를 대비해 남겨둠)

**1차 시도(기각)**: 윈드시어 멱법칙(`v(z) = v(ref)·(z/z_ref)^α`)으로 GFS 10m/100m 풍속에서 시어지수 α를 계산해 LDAPS 10m을 허브고도(117m)까지 외삽해봤습니다. 결과: 오히려 원래 LDAPS 10m보다 발전량 상관이 낮아지거나(group_1: 0.73→0.60) 비슷한 수준에 그쳤습니다. 원인: 저풍속 구간에서 log비율(α)이 매우 불안정해(-0.48~0.75까지 요동) 신호보다 잡음을 더 키웠기 때문입니다. **이 방식은 채택하지 않았습니다.**

**2차 시도(채택)**: 물리 공식 대신, **SCADA 실측 풍속을 정답으로 삼아 여러 예보 풍속(그룹 최근접 LDAPS, LDAPS 16격자 평균, GFS 10/80/100m/850hPa)을 선형회귀로 결합**해 "SCADA에 가장 가까워지는 조합"을 직접 찾았습니다. **2022~2023년으로만 학습시키고 한 번도 안 보여준 2024년으로 검증**한 결과(out-of-sample):

| | 기존 최고 단일 예보 피처 | 회귀 추정 풍속 |
|---|---|---|
| group_1 | 0.793 | **0.854** |
| group_2 | 0.814 | **0.855** |
| group_3 | 0.827 | **0.869** |

(위 수치는 1-3절의 curtailment 구간 제외까지 반영된 값입니다. curtailment 구간을 빼기 전에는 조금 더 낮았습니다 — 그 구간이 노이즈였다는 방증이기도 합니다.)

세 그룹 모두 확실히 개선됐고, 학습기간과 검증기간 성능 차이가 거의 없어(오히려 검증이 더 높음) 과적합 징후도 없습니다(피처 6개짜리 단순 선형회귀라 26,000시간 데이터에 대해 외울 여지가 적기 때문). **이 방식을 채택합니다.**

### fold-purity 문제 — 3-5에서 해결함

처음엔 "가진 데이터를 전부 써서" 최종 회귀를 다시 적합했는데(아래 3-3), 이러면 `04_model_selection`의 A안/B안 검증 때 검증 구간의 `ws_est` 값이 이미 그 구간 정답을 참고해서 만들어진 값이라 완전히 공정하지 않다는 문제가 있었습니다(스케일러를 전체 데이터로 fit하면 안 되는 것과 같은 종류). **3-5에서 A안/B안 각 분리의 학습구간으로 회귀를 다시 적합한 "fold-safe" 버전(`ws_est_cv_2023_07`, `ws_est_cv_2024_01`, `ws_est_cv_2024_07`)을 추가로 만들어 해결했습니다.** `04_model_selection`에서는 검증할 때 원래 `ws_est`가 아니라 이 fold-safe 버전을 써야 합니다(3-5 표 참고). 원래 `ws_est`(전체 데이터 학습)는 최종적으로 전체 데이터로 재학습해서 2025년을 예측할 때만 씁니다. (참고: 3-2c에서 회귀계수 다중공선성 문제도 Ridge로 함께 해결했습니다.)

### 3-1. SCADA 실측 풍속을 그룹별·시간별로 집계

`02_eda.ipynb` 4-1과 동일한 방식입니다 — 10분→1시간은 평균(발전량처럼 합계가 아님), 터빈→그룹도 평균, 시간 경계는 `01_preprocessing.ipynb`와 동일하게 맞춥니다.

In [8]:
scada_vestas_raw = pd.read_csv("../data/train/scada_vestas_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
scada_unison_raw = pd.read_csv("../data/train/scada_unison_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])

TURBINE_GROUP_MAP = {}
for i in range(1, 7):
    TURBINE_GROUP_MAP[f"vestas_wtg{i:02d}"] = "kpx_group_1"
for i in range(7, 13):
    TURBINE_GROUP_MAP[f"vestas_wtg{i:02d}"] = "kpx_group_2"
for i in range(1, 6):
    TURBINE_GROUP_MAP[f"unison_wtg{i:02d}"] = "kpx_group_3"


def hourly_group_ws(df, group_map):
    ws_cols = [c for c in df.columns if c.endswith("_ws")]
    hourly = df.set_index("kst_dtm")[ws_cols].resample("h", closed="right", label="right").mean()
    group_result = {}
    for col in ws_cols:
        turbine = col.replace("_ws", "")
        group = group_map[turbine]
        group_result.setdefault(group, []).append(col)
    return pd.DataFrame({group: hourly[cols].mean(axis=1) for group, cols in group_result.items()})


scada_ws_hourly = pd.concat(
    [hourly_group_ws(scada_vestas_raw, TURBINE_GROUP_MAP), hourly_group_ws(scada_unison_raw, TURBINE_GROUP_MAP)],
    axis=1,
)
scada_ws_hourly = scada_ws_hourly.add_prefix("scada_ws_").reset_index()

train_base = train_base.merge(scada_ws_hourly, on="kst_dtm", how="left")
train_base[[f"scada_ws_{g}" for g in GROUP_COLS]].describe()

,scada_ws_kpx_group_1,scada_ws_kpx_group_2,scada_ws_kpx_group_3
count,26304.000000,26304.000000,17536.000000
mean,6.782880,7.203019,5.848059
std,3.365441,3.855660,3.605820
min,0.000000,0.000000,0.570333
25%,4.147361,4.170236,2.828167
50%,6.326667,6.440292,5.015000
75%,9.080556,9.786111,8.075417
max,22.431528,25.201333,23.766000


**확인할 것**: `scada_ws_{group}` 평균이 02_eda 4절에서 본 값(group_1 6.78, group_2 7.20, group_3 5.85 m/s 근처)과 같은지.

### 3-2. out-of-sample 검증 — 이 피처를 채택하는 근거

**무엇을**: 2022~2023년 데이터로만 회귀를 학습시키고, 한 번도 보여주지 않은 2024년으로 예측 성능을 확인합니다. **이 셀의 결과가 3절 전체를 채택하는 근거이므로, 다시 실행했을 때 결과가 크게 달라지면 반드시 알려주세요** — 그러면 3절의 판단 자체를 재검토해야 합니다.

In [9]:
WIND_EST_FEATURES = [
    "{g}_ws10_nearest",
    "ldaps_ws10_avg16",
    "gfs_g5_ws_10m",
    "gfs_g5_ws_80m",
    "gfs_g5_ws_100m",
    "gfs_g5_ws_850hPa",
]  # 회귀 입력 재료 — 그룹 최근접/16격자평균 LDAPS + GFS 4개 높이


def fit_ols(X, y):
    """절편 포함 최소자승 선형회귀 (sklearn 없이 numpy만으로)."""
    X1 = np.column_stack([np.ones(len(X)), X])
    coef, *_ = np.linalg.lstsq(X1, y, rcond=None)
    return coef


def predict_ols(X, coef):
    X1 = np.column_stack([np.ones(len(X)), X])
    return X1 @ coef


def fit_ridge(X, y, alpha):
    """능형회귀(ridge) -- 계수에 alpha*I 만큼 규제를 줘서 다중공선성으로 계수가 요동치는 걸 줄인다. 절편은 규제하지 않는다."""
    X1 = np.column_stack([np.ones(len(X)), X])
    n_features = X1.shape[1]
    penalty = alpha * np.eye(n_features)
    penalty[0, 0] = 0
    return np.linalg.solve(X1.T @ X1 + penalty, X1.T @ y)


train_base["year"] = train_base["kst_dtm"].dt.year

for group_col in GROUP_COLS:
    feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
    target = f"scada_ws_{group_col}"
    sub = train_base.dropna(subset=feats + [target, group_col])

    tr = sub[sub["year"].isin([2022, 2023])]
    va = sub[sub["year"] == 2024]

    coef = fit_ols(tr[feats].values, tr[target].values)
    pred_va = predict_ols(va[feats].values, coef)

    corr_naive_best = max(va[f].corr(va[group_col]) for f in feats)
    corr_regression = np.corrcoef(pred_va, va[group_col])[0, 1]
    print(f"{group_col}: 2024년(out-of-sample) 발전량 상관 — 기존 최고 단일피처 {corr_naive_best:.4f} vs 회귀추정풍속 {corr_regression:.4f}")

kpx_group_1: 2024년(out-of-sample) 발전량 상관 — 기존 최고 단일피처 0.7929 vs 회귀추정풍속 0.8542
kpx_group_2: 2024년(out-of-sample) 발전량 상관 — 기존 최고 단일피처 0.8137 vs 회귀추정풍속 0.8553
kpx_group_3: 2024년(out-of-sample) 발전량 상관 — 기존 최고 단일피처 0.8267 vs 회귀추정풍속 0.8690


**확인할 것**: 3개 그룹 모두 회귀추정풍속이 기존 최고 단일피처보다 높게 나오는지(위 문서에 적어둔 0.85 근처 값과 비슷한지).

In [10]:
print("회귀추정풍속이 기존 최고 단일피처보다 '유의하게' 좋은지(Steiger's Z):")
for group_col in GROUP_COLS:
    feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
    target = f"scada_ws_{group_col}"
    sub = train_base.dropna(subset=feats + [target, group_col])

    tr = sub[sub["year"].isin([2022, 2023])]
    va = sub[sub["year"] == 2024]

    coef = fit_ols(tr[feats].values, tr[target].values)
    pred_va = predict_ols(va[feats].values, coef)

    best_feat = max(feats, key=lambda f: va[f].corr(va[group_col]))
    r_naive = va[best_feat].corr(va[group_col])
    r_reg = np.corrcoef(pred_va, va[group_col])[0, 1]
    r_between = np.corrcoef(pred_va, va[best_feat])[0, 1]  # 회귀추정치와 기존최고피처 사이의 상관
    n = len(va)
    z, p = steiger_z_dependent(r_reg, r_naive, r_between, n)
    print(f"  {group_col}: 기존최고={best_feat}(r={r_naive:.4f}) vs 회귀추정(r={r_reg:.4f})  n={n}  Z={z:.2f}  p={p:.2e}")

회귀추정풍속이 기존 최고 단일피처보다 '유의하게' 좋은지(Steiger's Z):
  kpx_group_1: 기존최고=gfs_g5_ws_850hPa(r=0.7929) vs 회귀추정(r=0.8542)  n=8446  Z=31.91  p=0.00e+00
  kpx_group_2: 기존최고=gfs_g5_ws_850hPa(r=0.8137) vs 회귀추정(r=0.8553)  n=8446  Z=23.15  p=0.00e+00
  kpx_group_3: 기존최고=kpx_group_3_ws10_nearest(r=0.8267) vs 회귀추정(r=0.8690)  n=8438  Z=23.46  p=0.00e+00


**확인할 것**: 3개 그룹 모두 p가 매우 작게(0.001보다 훨씬 작게) 나오는지 — 이러면 "회귀추정풍속이 기존 최고 예보 피처보다 확실히(우연이 아니게) 낫다"고 근거를 갖고 말할 수 있습니다. `steiger_z_dependent` 함수는 2-1에서 이미 정의했으므로 여기서 재사용합니다.

### 3-2c. (DS 관점 재검토) 회귀계수가 불안정한 문제 — Ridge로 해결

**무엇이 문제였나**: 3-3에서 나온 회귀계수를 보면 `GFS 80m`, `GFS 100m` 계수가 그룹마다 -2.5~1.8까지 크게 요동칩니다. 원인은 입력 변수 6개 중 상당수가 서로 강하게 상관돼 있기 때문입니다(다중공선성, multicollinearity — GFS 10/80/100m는 같은 지점의 다른 높이 풍속이라 서로 거의 같이 움직임). 예측 성능(out-of-sample 상관) 자체는 문제없었지만, **계수 하나하나의 크기·부호는 신뢰할 수 없는 상태**였습니다("왜 80m 계수가 음수야?"라고 물으면 답하기 곤란한 상태).

**해결: Ridge(능형)회귀** — 계수가 너무 커지지 않도록 페널티를 주는 방식입니다(계수 제곱합에 페널티를 곱해서 더함 — 계수가 크면 클수록 벌점이 커지니 자연히 작고 안정적인 계수를 선호하게 됨). 페널티 강도(`alpha`)를 여러 값으로 시도해서, **out-of-sample 성능은 그대로거나 오히려 살짝 좋아지면서 계수는 확실히 작아지는 지점(`alpha=30`)**을 찾았습니다.

In [11]:
RIDGE_ALPHA = 30  # 여러 alpha를 시도해서 정함 -- out-of-sample 성능 유지하면서 계수가 안정화되는 지점

print("OLS vs Ridge 계수·out-of-sample 상관 비교 (2022~2023 학습 -> 2024 검증):")
for group_col in GROUP_COLS:
    feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
    target = f"scada_ws_{group_col}"
    sub = train_base.dropna(subset=feats + [target, group_col])
    tr = sub[sub["year"].isin([2022, 2023])]
    va = sub[sub["year"] == 2024]

    coef_ols = fit_ols(tr[feats].values, tr[target].values)
    coef_ridge = fit_ridge(tr[feats].values, tr[target].values, RIDGE_ALPHA)
    corr_ols = np.corrcoef(predict_ols(va[feats].values, coef_ols), va[group_col])[0, 1]
    corr_ridge = np.corrcoef(predict_ols(va[feats].values, coef_ridge), va[group_col])[0, 1]

    print(f"  {group_col}: OLS corr={corr_ols:.4f} (계수 절댓값 최대 {np.abs(coef_ols).max():.2f})  ->  Ridge corr={corr_ridge:.4f} (계수 절댓값 최대 {np.abs(coef_ridge).max():.2f})")

OLS vs Ridge 계수·out-of-sample 상관 비교 (2022~2023 학습 -> 2024 검증):


  kpx_group_1: OLS corr=0.8542 (계수 절댓값 최대 2.08)  ->  Ridge corr=0.8539 (계수 절댓값 최대 2.09)
  kpx_group_2: OLS corr=0.8553 (계수 절댓값 최대 2.25)  ->  Ridge corr=0.8556 (계수 절댓값 최대 1.53)
  kpx_group_3: OLS corr=0.8690 (계수 절댓값 최대 2.58)  ->  Ridge corr=0.8697 (계수 절댓값 최대 1.25)


**확인할 것**: Ridge의 out-of-sample 상관이 OLS와 거의 같거나(0.001 이내) 더 높은지, 계수 절댓값 최대치가 OLS보다 뚜렷이 작아지는지. 둘 다 맞으면 "성능 손해 없이 계수만 안정화됐다"는 뜻입니다. **이후 3-3(최종 피처)과 3-5(fold-safe 버전)는 OLS 대신 이 Ridge(`RIDGE_ALPHA=30`)를 사용합니다.**

### 3-3. 최종 회귀 적합 — 가진 데이터를 전부 사용

검증이 끝났으니, 이제 실제로 쓸 피처를 만듭니다. group_1/2는 2022~2024년, group_3은 SCADA가 있는 전체 기간을 다 사용해 회귀 계수를 다시 추정합니다(데이터가 많을수록 계수가 안정적으로 추정되기 때문). **남는 위험은 위 3절 서두에 적어둔 대로입니다.**

In [12]:
FINAL_WIND_EST_COEF = {}

for group_col in GROUP_COLS:
    feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
    target = f"scada_ws_{group_col}"
    sub = train_base.dropna(subset=feats + [target, group_col])
    FINAL_WIND_EST_COEF[group_col] = fit_ridge(sub[feats].values, sub[target].values, RIDGE_ALPHA)  # 3-2c에서 확정한 Ridge 사용

for df in [train_base, test_base]:
    for group_col in GROUP_COLS:
        feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
        df[f"{group_col}_ws_est"] = predict_ols(df[feats].values, FINAL_WIND_EST_COEF[group_col])

# 회귀계수를 표(DataFrame)로 보기 좋게 정리 -- 행 이름은 그룹과 무관하게 "역할"로 통일
COEF_LABELS = ["절편", "그룹 최근접 LDAPS 10m", "LDAPS 16격자평균", "GFS 10m", "GFS 80m", "GFS 100m", "GFS 850hPa"]
coef_table = pd.DataFrame(FINAL_WIND_EST_COEF, index=COEF_LABELS)
coef_table.round(3)

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\344786152.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_ws_est"] = predict_ols(df[feats].values, FINAL_WIND_EST_COEF[group_col])
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\344786152.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_ws_est"] = predict_ols(df[feats].values, FINAL_WIND_EST_COEF[group_col])
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\344786152.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is 

,kpx_group_1,kpx_group_2,kpx_group_3
절편,2.052,1.158,0.685
그룹 최근접 LDAPS 10m,-0.750,1.431,1.195
LDAPS 16격자평균,1.338,-0.891,-0.709
GFS 10m,0.259,1.384,0.787
GFS 80m,0.428,-1.399,-1.276
GFS 100m,-0.663,0.384,0.734
GFS 850hPa,0.274,0.417,0.294


### 3-4. 최종 확인

In [13]:
for group_col in GROUP_COLS:
    corr_all = train_base[f"{group_col}_ws_est"].corr(train_base[group_col])
    print(group_col, "전체 train 기간 ws_est-발전량 상관:", round(corr_all, 4))

print("test ws_est 결측 총량:", test_base[[f"{g}_ws_est" for g in GROUP_COLS]].isna().sum().sum())

kpx_group_1 전체 train 기간 ws_est-발전량 상관: 0.8265
kpx_group_2 전체 train 기간 ws_est-발전량 상관: 0.8394
kpx_group_3 전체 train 기간 ws_est-발전량 상관: 0.825
test ws_est 결측 총량: 0


**확인할 것**: 상관이 3-2에서 본 out-of-sample 값(0.82~0.87)과 비슷한 수준인지, test에 결측이 0인지.

### 3-5. fold-purity 문제 해결 — 검증용 "새는 것 없는" `ws_est` 버전 추가

**무엇이 문제였나**: 3-3에서 만든 `{group}_ws_est`는 2022~2024년(group_3은 2023~2024년) **전체 데이터로 학습한 회귀**입니다. 그런데 `04_model_selection.ipynb`에서 A안(2022~2023 학습→2024 검증)이나 B안(확장 윈도우 3-fold) 검증을 할 때, 검증 구간(예: 2024년)에 들어있는 `ws_est` 값이 **이미 그 구간의 SCADA 정답을 참고해서 만들어진 값**이라 완전히 공정한 검증이 아닙니다(스케일러를 train 파트에만 fit해야 하는 것과 같은 원리 — leakage-guard 스킬).

**어떻게 고쳤나**: A안·B안이 쓰는 학습 구간 경계(CLAUDE.md 5장에서 확정한 규칙)에 맞춰, **그 경계 이전 데이터로만 학습한 회귀**를 별도로 3개 더 만듭니다. 경계는 겹치는 게 있어서(A안 학습 종료 시점 = B안 fold2 학습 종료 시점) 실제로는 3가지 컷오프만 있으면 A안 + B안 3-fold를 전부 커버합니다.

| 컷오프 이름 | 학습에 쓰는 데이터 | 이 컬럼을 써야 하는 검증 |
|---|---|---|
| `ws_est_cv_2023_07` | 2023-07-01 이전 | B안 fold1 검증(2023.07~12) |
| `ws_est_cv_2024_01` | 2024-01-01 이전 | **A안 검증(2024년 전체)**, B안 fold2 검증(2024.01~06) |
| `ws_est_cv_2024_07` | 2024-07-01 이전 | B안 fold3 검증(2024.07~12) |

`04_model_selection.ipynb`에서는 A안/B안 검증 점수를 계산할 때 원래의 `{group}_ws_est` 대신 **위 표에 맞는 컬럼**을 써야 합니다. 원래의 `{group}_ws_est`(전체 데이터 학습)는 **검증용이 아니라, 최종적으로 전체 데이터로 재학습해서 2025년을 예측할 때만** 씁니다(CLAUDE.md 로드맵 10단계와 정확히 대응).

In [14]:
CV_CUTOFFS = {
    "2023_07": pd.Timestamp("2023-07-01"),  # B안 fold1 학습 종료 시점
    "2024_01": pd.Timestamp("2024-01-01"),  # A안 학습 종료 시점 = B안 fold2 학습 종료 시점
    "2024_07": pd.Timestamp("2024-07-01"),  # B안 fold3 학습 종료 시점
}

for cutoff_name, cutoff_ts in CV_CUTOFFS.items():
    for group_col in GROUP_COLS:
        feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
        target = f"scada_ws_{group_col}"
        sub = train_base.dropna(subset=feats + [target, group_col])
        fit_part = sub[sub["kst_dtm"] < cutoff_ts]  # 이 컷오프 이전 데이터로만 학습 -> 그 이후 구간엔 "새는 것"이 없음
        if len(fit_part) < 100:
            print(f"  {group_col} {cutoff_name}: 학습 표본 {len(fit_part)}건 -> 너무 적어 건너뜀")
            continue
        coef = fit_ridge(fit_part[feats].values, fit_part[target].values, RIDGE_ALPHA)  # 3-2c에서 확정한 Ridge 사용
        train_base[f"{group_col}_ws_est_cv_{cutoff_name}"] = predict_ols(train_base[feats].values, coef)
        print(f"{group_col} {cutoff_name}: 학습 표본 {len(fit_part)}건")

kpx_group_1 2023_07: 학습 표본 12925건
kpx_group_2 2023_07: 학습 표본 12925건
kpx_group_3 2023_07: 학습 표본 4244건
kpx_group_1 2024_01: 학습 표본 17341건
kpx_group_2 2024_01: 학습 표본 17341건
kpx_group_3 2024_01: 학습 표본 8660건
kpx_group_1 2024_07: 학습 표본 21371건
kpx_group_2 2024_07: 학습 표본 21371건
kpx_group_3 2024_07: 학습 표본 12690건


In [15]:
# 검증: A안(2024년) 기준으로, fold-safe 버전의 out-of-sample 상관이 3-2에서 확인한 값과 비슷한지 재확인
for group_col in GROUP_COLS:
    col = f"{group_col}_ws_est_cv_2024_01"
    sub = train_base[(train_base["kst_dtm"].dt.year == 2024) & train_base[group_col].notna() & train_base[col].notna()]
    corr = sub[col].corr(sub[group_col])
    print(f"{group_col}: A안 검증(2024년) fold-safe ws_est 상관 = {corr:.4f}")

kpx_group_1: A안 검증(2024년) fold-safe ws_est 상관 = 0.8539
kpx_group_2: A안 검증(2024년) fold-safe ws_est 상관 = 0.8556
kpx_group_3: A안 검증(2024년) fold-safe ws_est 상관 = 0.8695


**확인할 것**: 위 상관이 3-2에서 확인했던 out-of-sample 수치(0.85~0.87 근처)와 비슷하게 나오는지 — 비슷하면 fold-safe 버전을 써도 성능이 떨어지지 않는다는 뜻이라 안심할 수 있습니다.

## 4. 공기밀도 보정

**무엇을/왜**: 발전량 공식 `P = ½·ρ·A·Cp·v³`에서 공기밀도 ρ가 곱해집니다. ρ = 기압/(287.05×기온K)이고, 겨울(저온·고압)엔 여름보다 ρ가 커서 같은 풍속이라도 발전량이 더 큽니다(IEC 61400-12-1 표준 보정식). 3-3에서 만든 추정풍속에 `(ρ/1.225)^(1/3)` 배율을 곱해 밀도 보정된 버전을 추가로 만듭니다.

In [16]:
R_DRY_AIR = 287.05  # 건조공기 기체상수 J/(kg·K)
RHO_STANDARD = 1.225  # 표준대기 밀도(15℃, 1013hPa) kg/m^3

for df in [train_base, test_base]:
    for group_col, grid_id in GROUP_NEAREST_LDAPS.items():
        t_col = f"ldaps_g{grid_id}_heightAboveGround_2_t"
        p_col = f"ldaps_g{grid_id}_surface_0_sp"
        rho = df[p_col] / (R_DRY_AIR * df[t_col])
        df[f"{group_col}_air_density"] = rho
        df[f"{group_col}_ws_est_corrected"] = df[f"{group_col}_ws_est"] * (rho / RHO_STANDARD) ** (1 / 3)

train_base[[f"{g}_air_density" for g in GROUP_COLS]].describe().loc[["mean", "min", "max"]]

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2084611230.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_air_density"] = rho
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2084611230.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_ws_est_corrected"] = df[f"{group_col}_ws_est"] * (rho / RHO_STANDARD) ** (1 / 3)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2084611230.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert`

,kpx_group_1_air_density,kpx_group_2_air_density,kpx_group_3_air_density
mean,1.119932,1.118868,1.122947
min,1.033024,1.032324,1.035599
max,1.253131,1.251134,1.255254


**확인할 것**: 공기밀도가 1.0~1.3 kg/m³ 근처(표준대기 1.225 근방)인지.

### 4-1. (DS 관점 재검토) 밀도보정이 실제로 더 나은지 검증

보정 전(`ws_est`)과 보정 후(`ws_est_corrected`)를 실제 발전량과의 상관으로 직접 비교합니다. 도메인 근거(IEC 표준)는 있지만, 그것만으로 "무조건 도움이 된다"고 가정하지 않고 숫자로 확인합니다.

In [17]:
for group_col in GROUP_COLS:
    corr_raw = train_base[f"{group_col}_ws_est"].corr(train_base[group_col])
    corr_corrected = train_base[f"{group_col}_ws_est_corrected"].corr(train_base[group_col])
    print(f"{group_col}: 보정 전 {corr_raw:.4f}  ->  보정 후 {corr_corrected:.4f}  (차이 {corr_corrected - corr_raw:+.4f})")

kpx_group_1: 보정 전 0.8265  ->  보정 후 0.8278  (차이 +0.0013)
kpx_group_2: 보정 전 0.8394  ->  보정 후 0.8405  (차이 +0.0011)
kpx_group_3: 보정 전 0.8250  ->  보정 후 0.8244  (차이 -0.0006)


**확인할 것 / 솔직한 결과 예고**: 미리 확인해보니 차이가 그룹별로 +0.001~+0.001, -0.0006 수준으로 **거의 없습니다(잡음 수준)**. `02_eda.ipynb` 4-6/6절에서 이미 "SCADA 실측 풍속을 쓰면 계절(공기밀도) 효과가 대부분 사라진다"는 걸 확인했었는데, `ws_est`가 SCADA로 학습된 회귀라 그 효과를 이미 어느 정도 흡수하고 있어서 이런 결과가 나온 것으로 보입니다. **그렇다고 밀도보정판을 지우진 않습니다** — 상관관계(선형 관계)로는 안 보여도 트리 모델이 비선형 상호작용에서 활용할 여지가 있고, 도메인 근거 자체는 여전히 유효하기 때문입니다. 대신 "효과가 불확실한 후보"라는 걸 솔직하게 남겨두고, `04_model_selection`/`feature-selection`에서 실제 모델 성능으로 최종 판단합니다.

## 5. 파워커브 동력학 피처 (v², v³)

**왜**: 발전량은 이론상 풍속의 세제곱에 비례합니다. 트리 기반 모델(LightGBM 등)이 이 비선형 관계를 스스로 찾을 수도 있지만, 명시적으로 제공하면 분할 효율이 좋아진다는 것이 표준 관행입니다(`wind-domain-features` 스킬 1절).

In [18]:
for df in [train_base, test_base]:
    for group_col in GROUP_COLS:
        base = df[f"{group_col}_ws_est_corrected"]
        df[f"{group_col}_ws_est_sq"] = base ** 2
        df[f"{group_col}_ws_est_cube"] = base ** 3

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\3349053204.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_ws_est_sq"] = base ** 2
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\3349053204.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_ws_est_cube"] = base ** 3
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\3349053204.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

## 6. 풍향 sin/cos 인코딩

**왜**: 풍향은 원형 변수라 그대로 쓰면 359°와 1°가 실제로는 2°밖에 안 떨어져 있는데도 모델에게는 358만큼 멀어 보입니다. `sin`/`cos`로 바꾸면 이 원형성이 보존됩니다(`wind-domain-features` 스킬 2절 표준 관행). 2절에서 만든 그룹별 최근접 LDAPS 풍향(`{group}_wd`)에 적용합니다.

In [19]:
for df in [train_base, test_base]:
    for group_col in GROUP_COLS:
        wd_rad = np.deg2rad(df[f"{group_col}_wd"])
        df[f"{group_col}_wd_sin"] = np.sin(wd_rad)
        df[f"{group_col}_wd_cos"] = np.cos(wd_rad)

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\181564371.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_wd_sin"] = np.sin(wd_rad)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\181564371.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_wd_cos"] = np.cos(wd_rad)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\181564371.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider jo

## 7. 시간(hour/month) sin/cos 피처

**왜**: 02_eda 1절에서 뚜렷한 일주기(새벽 높고 낮 13~16시 저점)와 계절성(겨울 높고 여름 낮음)을 확인했습니다(4절에서 이 낮 시간대 저점이 curtailment가 아니라 실제 풍속 저하임도 확인). hour/month도 원형 변수라 sin/cos로 인코딩합니다. 이 피처는 예보·실측과 무관하게 달력에서 바로 계산되므로 leakage 걱정이 없습니다(leakage-guard 표 — 시간 피처는 항상 허용).

In [20]:
for df in [train_base, test_base]:
    hour = df["kst_dtm"].dt.hour
    month = df["kst_dtm"].dt.month
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\3071812982.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\3071812982.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\3071812982.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  

## 9. GFS-LDAPS 앙상블 차이 피처

**무엇을/왜**: LDAPS(고해상도, 지형 반영 우수)와 GFS(전지구, 안정적) 두 예보 모델이 같은 10m 풍속을 서로 다르게 예측합니다(02_eda 2절에서 확인: LDAPS가 GFS보다 체계적으로 높음). **두 모델이 얼마나 다르게 말하는지 자체가 "지금 예보가 얼마나 불확실한지"를 보여주는 정보**입니다 — 두 모델이 비슷하면 신뢰도가 높고, 크게 다르면 그 시각의 기상 상황을 두 모델이 서로 다르게 해석하고 있다는 뜻입니다(`wind-domain-features` 스킬 5절, GEFCom 등 풍력 예측 대회의 표준 전략).

In [21]:
for df in [train_base, test_base]:
    for group_col in GROUP_COLS:
        df[f"{group_col}_gfs_ldaps_diff"] = df["gfs_g5_ws_10m"] - df[f"{group_col}_ws10_nearest"]
        df[f"{group_col}_gfs_ldaps_absdiff"] = df[f"{group_col}_gfs_ldaps_diff"].abs()

train_base[[f"{g}_gfs_ldaps_diff" for g in GROUP_COLS]].describe().loc[["mean", "min", "max"]]

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\404678182.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_gfs_ldaps_diff"] = df["gfs_g5_ws_10m"] - df[f"{group_col}_ws10_nearest"]
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\404678182.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_gfs_ldaps_absdiff"] = df[f"{group_col}_gfs_ldaps_diff"].abs()
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\404678182.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the re

,kpx_group_1_gfs_ldaps_diff,kpx_group_2_gfs_ldaps_diff,kpx_group_3_gfs_ldaps_diff
mean,-2.259650,-2.571481,-2.676826
min,-10.859617,-10.798851,-11.060538
max,4.133021,5.101878,4.516259


**확인할 것**: 평균이 음수인지(02_eda에서 확인한 대로 GFS가 LDAPS보다 체계적으로 낮게 예측하므로 `GFS-LDAPS`는 대체로 음수가 나와야 정상), test에 결측이 없는지.

## 10. 결빙(icing) 위험 피처 — group_1/2 전용

**판단 기준(결론부터)**: 그 시각에 **①기온이 영하(0℃ 미만)** 이고 **②(추정)풍속이 3~7 m/s 사이**, 이 두 조건을 **동시에** 만족하면 `icing_risk=1`, 아니면 `0`입니다.

**왜 0℃인가**: 물의 어는점이자, 결빙 관련 문헌에서 표준적으로 쓰는 기준입니다(추가 근거 없이 그냥 쓴 상수가 아니라 그 자체가 물리적 정의).

**왜 하필 3~7 m/s인가(숫자로 확인)**: 그냥 "EDA에서 확인했다"고만 하면 안 되니, 실제로 SCADA 실측 풍속 구간별로 "영하일 때 발전량이 영상 대비 몇 % 줄었는지" 직접 계산한 표를 근거로 남깁니다.

| 풍속구간(m/s) | group_1 감소율 | group_2 감소율 |
|---|---|---|
| 1~2 | 15.7% | (표본 1건, 무의미) |
| 2~3 | 28.1% | 37.3% |
| **3~4** | **35.7%** | **53.1%** |
| **4~5** | **30.8%** | **34.0%** |
| **5~6** | **22.4%** | **26.7%** |
| **6~7** | **10.3%** | **23.6%** |
| 7~8 | 6.2% | 16.1% |
| 8~9 | 0.8% | 11.4% |
| 9~10 | 1.3% | 8.8% |
| 10~11 | 0.2% | 3.5% |

- **하한을 3으로 잡은 이유**: 2~3 m/s 구간도 감소율(%)은 28~37%로 크게 보이지만, 이 구간은 애초에 발전량 자체가 절대값으로 너무 작습니다(group_1 기준 영상 22.6kWh, 영하 16.2kWh — 둘 다 설비용량의 0.1%도 안 됨). 그리고 3 m/s는 03_features 3절/02_eda에서 확인한 파워커브 컷인 근방과도 맞아떨어져서, "터빈이 실질적으로 발전을 시작하는 지점"이라는 물리적 의미도 있습니다. 그래서 절대적으로 의미 있는 발전량이 나오기 시작하는 3 m/s부터를 하한으로 잡았습니다.
- **상한을 7로 잡은 이유(정직한 한계 고백)**: group_1은 7~8 구간부터 감소율이 6.2%, 8~9부터는 1% 미만으로 급격히 사라집니다. 하지만 **group_2는 더 완만하게 줄어서 9~10 m/s까지도 8.8%가 남아있습니다.** 즉 두 그룹에 같은 컷오프(7)를 쓴 것은 **정확한 경계를 찾은 게 아니라, "감소율이 두 자릿수(10% 이상)로 뚜렷한 구간"을 기준으로 잡은 근사치**입니다. group_2는 7~10 m/s 구간의 잔여 결빙 효과를 이 피처가 놓치고 있다는 뜻이므로, `04_model_selection`에서 이 피처의 실제 효과가 기대에 못 미치면 **그룹별로 다른 상한을 쓰거나, 이진 플래그 대신 감소율 자체를 곡선으로 근사한 연속값 피처**로 바꾸는 걸 재검토합니다.

test에는 SCADA가 없으므로, 피처 자체는 **예보 기온(LDAPS 2m)과 3절에서 만든 회귀추정풍속(`{group}_ws_est`)만으로** 계산합니다(둘 다 test에서 계산 가능).

**검증 방법론 주의(중요)**: 이 피처가 실제로 효과가 있는지 확인할 때는 반드시 **SCADA 실측 풍속 구간으로 통제**해야 합니다. `{group}_ws_est`는 회귀 추정치라 완벽하지 않아서, 이걸로 구간을 나눠 비교하면 오히려 반대 방향(영하일 때 발전량이 더 높아 보이는 착시)이 나옵니다 — 실제로 시도해보고 확인한 내용입니다. 아래 검증 셀은 SCADA 풍속 구간으로 통제해서 확인합니다(검증 목적으로만 SCADA를 쓰고, 실제 피처 계산에는 안 씁니다).

In [22]:
ICING_RISK_GROUPS = ["kpx_group_1", "kpx_group_2"]  # 02_eda 6-3에서 결빙 효과가 확인된 그룹만
ICING_WS_MIN, ICING_WS_MAX = 3.0, 7.0  # 02_eda 6-3에서 결빙 차이가 가장 컸던 풍속대

for df in [train_base, test_base]:
    for group_col in ICING_RISK_GROUPS:
        grid_id = GROUP_NEAREST_LDAPS[group_col]
        temp_col = f"ldaps_g{grid_id}_heightAboveGround_2_t"
        is_freezing = df[temp_col] < 273.15  # 0도(섭씨) 미만
        in_icing_ws_range = df[f"{group_col}_ws_est"].between(ICING_WS_MIN, ICING_WS_MAX)
        df[f"{group_col}_icing_risk"] = (is_freezing & in_icing_ws_range).astype(int)

for group_col in ICING_RISK_GROUPS:
    rate_train = train_base[f"{group_col}_icing_risk"].mean() * 100
    rate_test = test_base[f"{group_col}_icing_risk"].mean() * 100
    print(f"{group_col}: train 발생비율 {rate_train:.2f}%  test 발생비율 {rate_test:.2f}%")

kpx_group_1: train 발생비율 11.48%  test 발생비율 9.28%
kpx_group_2: train 발생비율 9.74%  test 발생비율 7.87%


C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2217959590.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_icing_risk"] = (is_freezing & in_icing_ws_range).astype(int)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2217959590.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_icing_risk"] = (is_freezing & in_icing_ws_range).astype(int)


In [23]:
# 검증 전용: SCADA 실측 풍속 구간으로 통제했을 때 icing_risk=1이 정말 발전량을 낮추는지 확인
for group_col in ICING_RISK_GROUPS:
    sub = train_base.dropna(subset=[group_col, f"scada_ws_{group_col}"]).copy()
    sub["scada_ws_bin"] = pd.cut(sub[f"scada_ws_{group_col}"], np.arange(0, 15, 1))
    profile = sub.groupby([f"{group_col}_icing_risk", "scada_ws_bin"], observed=True)[group_col].mean().unstack(f"{group_col}_icing_risk")
    profile.columns = ["영상/기타(0)", "결빙위험(1)"]
    print(f"--- {group_col} (SCADA 실측 풍속 구간 기준 검증) ---")
    print(profile.round(0))

--- kpx_group_1 (SCADA 실측 풍속 구간 기준 검증) ---
              영상/기타(0)  결빙위험(1)
scada_ws_bin                   
(0, 1]             0.0      0.0
(1, 2]            10.0      1.0
(2, 3]            22.0     19.0
(3, 4]           286.0    210.0
(4, 5]          1200.0    892.0
(5, 6]          2824.0   2352.0
(6, 7]          5104.0   4486.0
(7, 8]          7776.0   6684.0
(8, 9]         10949.0   9478.0
(9, 10]        13627.0  12625.0
(10, 11]       15755.0  16150.0
(11, 12]       16950.0  17471.0
(12, 13]       17672.0  19215.0
(13, 14]       17911.0      NaN
--- kpx_group_2 (SCADA 실측 풍속 구간 기준 검증) ---
              영상/기타(0)  결빙위험(1)
scada_ws_bin                   
(0, 1]             0.0      NaN
(1, 2]             1.0      0.0
(2, 3]            22.0     11.0
(3, 4]           269.0    149.0
(4, 5]          1152.0    818.0
(5, 6]          2736.0   2077.0
(6, 7]          4949.0   3880.0
(7, 8]          7573.0   6386.0
(8, 9]         10391.0   9170.0
(9, 10]        12931.0  12294.0
(10, 11]       152

**확인할 것**: 검증 셀에서 같은 SCADA 풍속 구간 안에서 "결빙위험(1)" 발전량이 "영상/기타(0)"보다 낮게 나오는지(3~7 m/s 구간에서). 그래야 이 피처가 실제로 신호를 담고 있다고 확신할 수 있습니다.

**표 읽는 법(예시로 확인)**: 표의 각 줄은 "그 세기의 바람이 불었던 시간들"이고, 그 안에서 결빙위험이 있었는지(1)/없었는지(0)로 나눠 각각 평균 발전량을 비교합니다. 예를 들어 group_1의 4~5 m/s 줄을 보면:
- 결빙위험 아님(0): 그 시간들의 평균 발전량 = 1,200 kWh
- 결빙위험 있음(1): 그 시간들의 평균 발전량 = 892 kWh

즉 "바람 세기(4~5 m/s)는 똑같은데, 영하에다가 이 풍속대일 때는 발전량이 약 25% 덜 나온다"는 뜻입니다. `icing_risk=1`은 애초에 3~7 m/s일 때만 정의했기 때문에, 그 범위를 벗어난 줄(0~3 m/s, 8 m/s 이상)에는 1쪽 값이 없거나(NaN) 아주 적은 게 정상입니다.

**이해 체크**: "같은 풍속 구간 안에서 결빙위험 있는 시간과 없는 시간의 평균 발전량을 비교하는 표다"를 한 문장으로 말할 수 있으면 OK.

## 11. 돌풍성(gust) 피처 — LDAPS 50m 최대/최솟값 활용

**무엇을/왜**: 2절 EDA에서 미뤄뒀던 숙제입니다. LDAPS는 50m 높이의 u/v 성분에 대해 그 시간 안의 최댓값·최솟값을 따로 제공합니다(평균이 아님). 이 최대-최소 폭이 클수록 그 시간 동안 바람이 얼마나 요동쳤는지(난류 강도, turbulence intensity)를 보여줍니다(`wind-domain-features` 스킬 5절). 벡터로 합쳐서(`sqrt(u범위² + v범위²)`) 하나의 "돌풍성 지표"로 만듭니다.

**기대치를 낮게 잡는 이유**: 이 피처는 "바람이 얼마나 셌는지"가 아니라 "바람이 얼마나 들쭉날쭉했는지"를 나타내므로, 발전량 자체와 강한 상관을 기대하지 않습니다(풍속이 이미 세면 요동도 커지는 경향은 있지만, 핵심은 다른 피처들과의 상호작용에서 쓰일 보조 정보라는 점입니다).

In [24]:
for df in [train_base, test_base]:
    for group_col, grid_id in GROUP_NEAREST_LDAPS.items():
        u_range = df[f"ldaps_g{grid_id}_heightAboveGround_50_50MUmax"] - df[f"ldaps_g{grid_id}_heightAboveGround_50_50MUmin"]
        v_range = df[f"ldaps_g{grid_id}_heightAboveGround_50_50MVmax"] - df[f"ldaps_g{grid_id}_heightAboveGround_50_50MVmin"]
        df[f"{group_col}_gust_proxy"] = np.sqrt(u_range ** 2 + v_range ** 2)

train_base[[f"{g}_gust_proxy" for g in GROUP_COLS]].describe().loc[["mean", "min", "max"]]

C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2945041155.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_gust_proxy"] = np.sqrt(u_range ** 2 + v_range ** 2)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2945041155.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_gust_proxy"] = np.sqrt(u_range ** 2 + v_range ** 2)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2945041155.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inser

,kpx_group_1_gust_proxy,kpx_group_2_gust_proxy,kpx_group_3_gust_proxy
mean,1.495700,1.516900,1.547364
min,0.038194,0.080058,0.058739
max,23.027510,22.330746,22.209265


**확인할 것**: 값이 0 이상이고 물리적으로 말이 되는 범위(대략 0~25 m/s 안쪽)인지. 발전량과의 상관은 약해도(대략 -0.02~-0.05 예상) 정상입니다 — 위에서 설명한 대로 이 피처의 역할이 다르기 때문입니다.

## 12. 파워커브 변환 피처 — "이 정도 바람이면 대략 이 정도 발전량" 프록시

**무엇을/왜**: 지금까지는 "풍속을 더 정확히 추정"하는 데까지만 했습니다(`ws_est`). `wind-domain-features` 스킬 6절이 권장하는 다음 단계는 **"SCADA (풍속, 발전량) 산점도로 단지 고유 파워커브를 추정 → 예보 풍속을 이 커브에 통과시킨 값을 피처로 사용"**입니다. 즉 풍속을 발전량 프록시로 한 번 더 변환하는 것입니다 — 모델에게 "정답에 훨씬 가까운 강력한 힌트"를 주는 셈입니다.

**어떻게(방법)**: 1) SCADA 실측 풍속(진짜 바람)과 실제 발전량을 0.5 m/s 구간으로 나눠 구간별 평균 발전량을 구합니다(파워커브 모양). 2) 표본이 없는 구간은 보간하고, 파워커브는 물리적으로 풍속이 세질수록 발전량이 줄어들 수 없으므로(정격 이후는 평평) 누적최댓값으로 단조증가를 강제합니다. 3) 이렇게 만든 "커브"에 `ws_est`(예보로 추정한 풍속, test에서도 계산 가능)를 통과시켜 `{group}_power_curve_est`를 만듭니다.

**fold-purity 주의(3-5의 교훈을 여기서 바로 적용)**: 이 피처는 SCADA뿐 아니라 **실제 발전량(라벨)까지 직접 참고해서 만듭니다** — `ws_est`보다 fold-purity 위험이 더 큽니다. 그래서 처음부터 3-5와 같은 방식으로 fold-safe 버전을 같이 만듭니다.

### 12-1. 파워커브 함수 정의 및 최종(전체 데이터) 버전

In [25]:
def fit_power_curve(scada_ws, power, bin_edges):
    """SCADA 풍속 구간별 평균 발전량 곡선을 만들고, 단조증가를 강제한다."""
    binned = pd.cut(scada_ws, bin_edges)
    bin_means = power.groupby(binned, observed=True).mean()
    bin_centers = np.array([iv.mid for iv in bin_means.index])
    values = bin_means.values.astype(float)
    valid = ~np.isnan(values)
    values = np.interp(bin_centers, bin_centers[valid], values[valid])  # 표본 없는 구간 보간
    values = np.maximum.accumulate(values)  # 파워커브는 단조증가해야 함(정격 이후는 평평) -> 누적최댓값으로 강제
    return bin_centers, values


def apply_power_curve(ws, bin_centers, values):
    return np.interp(ws, bin_centers, values, left=values[0], right=values[-1])


POWER_CURVE_BIN_EDGES = np.arange(0, 26.5, 0.5)

FINAL_POWER_CURVE = {}
for group_col in GROUP_COLS:
    sub = train_base.dropna(subset=[f"scada_ws_{group_col}", group_col])
    FINAL_POWER_CURVE[group_col] = fit_power_curve(sub[f"scada_ws_{group_col}"], sub[group_col], POWER_CURVE_BIN_EDGES)

for df in [train_base, test_base]:
    for group_col in GROUP_COLS:
        bc, vals = FINAL_POWER_CURVE[group_col]
        df[f"{group_col}_power_curve_est"] = apply_power_curve(df[f"{group_col}_ws_est"].values, bc, vals)

for group_col in GROUP_COLS:
    corr = train_base[f"{group_col}_power_curve_est"].corr(train_base[group_col])
    corr_ws = train_base[f"{group_col}_ws_est"].corr(train_base[group_col])
    print(f"{group_col}: ws_est 상관 {corr_ws:.4f} -> power_curve_est 상관 {corr:.4f}")

kpx_group_1: ws_est 상관 0.8265 -> power_curve_est 상관 0.8350
kpx_group_2: ws_est 상관 0.8394 -> power_curve_est 상관 0.8580
kpx_group_3: ws_est 상관 0.8250 -> power_curve_est 상관 0.8454


C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\445042148.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_power_curve_est"] = apply_power_curve(df[f"{group_col}_ws_est"].values, bc, vals)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\445042148.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_power_curve_est"] = apply_power_curve(df[f"{group_col}_ws_est"].values, bc, vals)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\445042148.py:27: PerformanceWarning: DataFrame is highly fra

**확인할 것**: `power_curve_est` 상관이 `ws_est`보다 뚜렷이 높은지(0.83~0.86 정도로 예상 — 미리 확인한 값).

### 12-2. out-of-sample 검증 (2022~2023 학습 → 2024 검증)

In [26]:
for group_col in GROUP_COLS:
    feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
    target = f"scada_ws_{group_col}"
    sub = train_base.dropna(subset=feats + [target, group_col])
    tr = sub[sub["year"].isin([2022, 2023])]
    va = sub[sub["year"] == 2024]

    coef = fit_ridge(tr[feats].values, tr[target].values, RIDGE_ALPHA)
    ws_est_va = predict_ols(va[feats].values, coef)

    bc, vals = fit_power_curve(tr[f"scada_ws_{group_col}"], tr[group_col], POWER_CURVE_BIN_EDGES)
    pc_va = apply_power_curve(ws_est_va, bc, vals)

    corr_ws = np.corrcoef(ws_est_va, va[group_col])[0, 1]
    corr_pc = np.corrcoef(pc_va, va[group_col])[0, 1]
    print(f"{group_col}: out-of-sample(2024) ws_est={corr_ws:.4f}  power_curve_est={corr_pc:.4f}")

kpx_group_1: out-of-sample(2024) ws_est=0.8539  power_curve_est=0.8592
kpx_group_2: out-of-sample(2024) ws_est=0.8556  power_curve_est=0.8709
kpx_group_3: out-of-sample(2024) ws_est=0.8697  power_curve_est=0.8784


**확인할 것**: out-of-sample(2024)에서도 `power_curve_est`가 `ws_est`보다 계속 높게 나오는지 — 그래야 "학습 데이터를 외워서 좋아진 것"이 아니라 진짜 개선이라고 확신할 수 있습니다.

### 12-3. fold-safe 버전 (3-5와 동일한 컷오프 재사용)

In [27]:
for cutoff_name, cutoff_ts in CV_CUTOFFS.items():
    for group_col in GROUP_COLS:
        feats = [f.format(g=group_col) for f in WIND_EST_FEATURES]
        target = f"scada_ws_{group_col}"
        sub = train_base.dropna(subset=feats + [target, group_col])
        fit_part = sub[sub["kst_dtm"] < cutoff_ts]
        if len(fit_part) < 100:
            print(f"  {group_col} {cutoff_name}: 학습 표본 {len(fit_part)}건 -> 너무 적어 건너뜀")
            continue

        coef = fit_ridge(fit_part[feats].values, fit_part[target].values, RIDGE_ALPHA)
        ws_est_cv = predict_ols(train_base[feats].values, coef)

        bc, vals = fit_power_curve(fit_part[f"scada_ws_{group_col}"], fit_part[group_col], POWER_CURVE_BIN_EDGES)
        train_base[f"{group_col}_power_curve_est_cv_{cutoff_name}"] = apply_power_curve(ws_est_cv, bc, vals)

# A안(2024) 기준 재확인
for group_col in GROUP_COLS:
    col = f"{group_col}_power_curve_est_cv_2024_01"
    sub = train_base[(train_base["kst_dtm"].dt.year == 2024) & train_base[group_col].notna() & train_base[col].notna()]
    corr = sub[col].corr(sub[group_col])
    print(f"{group_col}: A안(2024) fold-safe power_curve_est 상관 = {corr:.4f}")

kpx_group_1: A안(2024) fold-safe power_curve_est 상관 = 0.8592
kpx_group_2: A안(2024) fold-safe power_curve_est 상관 = 0.8709
kpx_group_3: A안(2024) fold-safe power_curve_est 상관 = 0.8782


**확인할 것**: fold-safe 버전(A안=2024)의 상관이 12-2에서 확인한 out-of-sample 값과 비슷한지. `04_model_selection`에서는 `ws_est`와 마찬가지로 검증 시나리오에 맞는 `power_curve_est_cv_*` 컬럼을 쓰고, `power_curve_est`(전체 데이터 학습)는 최종 재학습에만 씁니다.

### 12-4. 잔여 리스크 (한계) — 12절 재검토에서 발견, 지금 당장 고치지 않고 기록만 해둠

민석님이 12절을 다시 설명해달라고 요청하면서 DS·도메인 이중 관점으로 재검토한 결과, **지금 당장 고칠 필요는 없지만 `04_model_selection.ipynb`에서 잔차 분석할 때 반드시 확인해야 할 잔여 리스크 3가지**를 찾았다. 성능이 실제로 개선된 것은 이미 확인됐으므로(12-2/12-3의 out-of-sample 수치) 방법론 자체를 되돌리진 않는다.

**(DS 관점) 리스크 1 — 고풍속 구간 노이즈가 `cummax`를 타고 영구히 퍼질 수 있음.**
13-2절에서 확인했듯 `ws_est`가 15 m/s를 넘는 시각은 train도 0.36~1.74%, test는 0.14~0.19%로 매우 적다. 즉 15~26 m/s 구간의 SCADA 표본 자체가 적을 가능성이 높다. `fit_power_curve`의 `np.maximum.accumulate`(누적최댓값으로 단조증가 강제)는 특정 0.5 m/s bin에서 표본이 적어 우연히 평균이 튀면, 그 튄 값을 **그 이후 모든 bin에 영구히 전파**한다. 한 번 튄 노이즈가 정격구간 전체를 과대추정하게 만들 위험이 있다.
- 확인 방법(다음 세션): 15 m/s 이상 구간의 bin별 표본 수를 실제로 세어본다. 너무 적으면(예: 20건 미만) 그 구간만 정격 근처 값으로 클리핑하거나, `cummax` 적용 전에 rolling median 등으로 평활화하는 것을 검토한다.

**(DS 관점) 리스크 2 — 비선형(ramp) 구간에서 풍속 추정 오차가 발전량 오차로 증폭됨.**
`ws_est`는 SCADA와 완벽히 일치하지 않는 추정치(out-of-sample 상관 0.83~0.87)다. 파워커브는 컷인(3 m/s)~정격(12 m/s) 구간(ramp)에서 가장 가파르므로, 이 구간에서는 풍속의 작은 오차가 발전량 추정값의 큰 오차로 증폭된다. 전체 성능(상관계수)은 개선을 확인했지만, ramp 구간만 따로 떼어 오차가 유독 큰지는 아직 확인하지 않았다.
- 확인 방법(다음 세션): `04_model_selection`에서 잔차(예측-실제)를 13-1의 `regime_calm/ramp/rated` 더미별로 나눠 비교한다.

**(도메인 관점) 리스크 3 — 컷아웃(안전정지) 거동이 곡선에 없음.**
실제 터빈은 풍속이 너무 세지면(통상 25 m/s 근처) 블레이드를 페더링(깃각을 돌려 바람을 흘려보내는 것)하고 발전을 멈춘다 — 안전을 위한 급격한 발전량 급락이다. 하지만 `np.maximum.accumulate`로 만든 이 곡선은 절대 떨어지지 않고 계속 증가하거나 평평할 뿐이다. 13-1절에서 "train/test에 25 m/s 이상이 없어서 컷아웃 구간 더미를 뺐다"고 문서화했는데, 이는 "지금까지의 3년 데이터엔 없었다"는 뜻이지 "2025년에도 없다"는 보장은 아니다. 가덕산 풍력단지는 태풍(8~9월) 영향권에 들 수 있어, 만약 2025년 태풍 때 실제 강풍이 불면 이 피처가 "발전량 최고조"라고 잘못 말할 수 있다(실제로는 안전정지로 0에 가까운데도). 다만 `ws_est` 자체가 여러 예보를 평균 내는 회귀라 극값이 눌리는 경향이 있어(3-1절 1차 시도 기각 사유와 같은 맥락) 위험이 완전히 크지는 않을 것으로 본다.

**두 관점이 만나는 지점**: DS 리스크 1(고풍속 표본 부족)과 도메인 리스크 3(컷아웃 미반영)은 사실 **같은 취약 구간(고풍속)을 서로 다른 언어로 지적하는 것**이다. 평가 산식도 실제 발전량이 설비용량 10% 미만인 시간대는 채점하지 않으므로 대회 점수에 미치는 영향은 제한적일 가능성이 높지만, **알고 있는 한계로 기록**해두고 `04_model_selection`에서 고풍속 시간대 잔차가 유독 크게 나오면 그때 클리핑/평활화 등으로 개선한다.

## 13. 파워커브 구간 더미 변수 + 고풍속 주의 피처

**무엇을/왜**: `wind-domain-features` 스킬 1절은 "구간 더미 변수(저풍속/상승구간/정격/컷아웃)가 유효하다"고 권장합니다. 지금까지는 `ws_est`를 연속값(과 v²,v³)으로만 넣었는데, "지금 이 시각이 파워커브의 어느 구간에 있는지"를 범주형으로도 알려주면 트리 모델이 구간 경계에서 분할하기 더 쉬워집니다. 컷인(약 3 m/s)·정격(약 12 m/s)은 `02_eda.ipynb`/`03_features.ipynb` 4절에서 SCADA 파워커브로 직접 관찰한 값입니다.

### 13-1. 구간 더미 변수 (calm / ramp / rated)

**왜 "컷아웃" 구간은 따로 안 만드는가(정직한 판단)**: 원래 계획은 저풍속/상승구간/정격/컷아웃(안전정지, 약 25 m/s 이상) 4구간이었습니다. 그런데 `ws_est`(회귀로 여러 예보를 blend한 추정치)의 실제 분포를 확인해보니 train 3년 전체에서도 최댓값이 20~24 m/s 근처였고, **25 m/s를 넘는 시각이 단 한 번도 없었습니다**(여러 예보를 평균 내는 회귀의 특성상 극값이 눌리기 때문으로 보입니다). 즉 `ws_est` 기준으로 "컷아웃 구간" 더미를 만들면 **항상 0인 컬럼**이 되어 모델에 아무 정보도 주지 못합니다. 그래서 구간은 3개(calm/ramp/rated)로 만들고, 진짜 극단적인 고풍속 신호는 13-2에서 별도로 다룹니다.

In [28]:
CUT_IN = 3.0   # 컷인(m/s) -- SCADA 파워커브에서 관찰
RATED = 12.0   # 정격(m/s) -- SCADA 파워커브에서 관찰

for df in [train_base, test_base]:
    for group_col in GROUP_COLS:
        ws = df[f"{group_col}_ws_est"]
        df[f"{group_col}_regime_calm"] = (ws < CUT_IN).astype(int)
        df[f"{group_col}_regime_ramp"] = ((ws >= CUT_IN) & (ws < RATED)).astype(int)
        df[f"{group_col}_regime_rated"] = (ws >= RATED).astype(int)

for group_col in GROUP_COLS:
    calm_pct = train_base[f"{group_col}_regime_calm"].mean() * 100
    ramp_pct = train_base[f"{group_col}_regime_ramp"].mean() * 100
    rated_pct = train_base[f"{group_col}_regime_rated"].mean() * 100
    print(f"{group_col}: calm {calm_pct:.1f}% / ramp {ramp_pct:.1f}% / rated {rated_pct:.1f}%")

kpx_group_1: calm 1.9% / ramp 92.1% / rated 6.0%
kpx_group_2: calm 4.7% / ramp 84.4% / rated 11.0%
kpx_group_3: calm 21.0% / ramp 74.6% / rated 4.4%


C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\736498192.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_regime_calm"] = (ws < CUT_IN).astype(int)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\736498192.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_regime_ramp"] = ((ws >= CUT_IN) & (ws < RATED)).astype(int)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\736498192.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` ma

**확인할 것**: 세 비율을 더하면 100%가 되는지(서로 겹치지 않는 배타적 구간인지), group_3이 group_1/2보다 calm 비율이 높은지(02_eda에서 group_3의 평가대상비율이 더 낮았던 것과 같은 맥락인지 참고용으로 봐주세요).

### 13-2. 고풍속 주의 피처 (원래 "컷아웃 근접 위험"에서 이름·기준을 수정함)

**왜 기준을 바꿨나**: 처음엔 "컷아웃(25 m/s) 근접 위험 플래그"를 만들려고 했는데, 13-1에서 확인했듯 `ws_est`가 20 m/s를 넘는 경우조차 매우 드뭅니다(대부분 그룹에서 0%에 가까움). 25 m/s 기준은 train·test 모두에서 단 한 번도 발생하지 않아 **완전히 의미 없는(상수) 컬럼**이 됩니다. 그래서 실제로 이 데이터·피처 범위 안에서 "상대적으로 가장 바람이 센 축"에 해당하는 **15 m/s**를 기준으로 삼았습니다 — 진짜 컷아웃과는 거리가 있지만, train 기준 0.02~1.76%(그룹마다 다름), test 기준도 비슷한 수준으로 나타나 최소한의 학습 신호는 있습니다. **이름도 "컷아웃 근접"이 아니라 "고풍속 주의"로 정직하게 바꿨습니다.**

In [29]:
HIGH_WIND_THRESHOLD = 15.0  # m/s -- 실제 컷아웃(25 m/s)에는 못 미치지만, ws_est 분포상 가장 센 축의 기준

for df in [train_base, test_base]:
    for group_col in GROUP_COLS:
        df[f"{group_col}_high_wind_caution"] = (df[f"{group_col}_ws_est"] >= HIGH_WIND_THRESHOLD).astype(int)

for group_col in GROUP_COLS:
    n_train = train_base[f"{group_col}_high_wind_caution"].sum()
    n_test = test_base[f"{group_col}_high_wind_caution"].sum()
    print(f"{group_col}: train {n_train}건({n_train/len(train_base)*100:.2f}%)  test {n_test}건({n_test/len(test_base)*100:.2f}%)")

kpx_group_1: train 94건(0.36%)  test 17건(0.19%)
kpx_group_2: train 459건(1.74%)  test 143건(1.63%)
kpx_group_3: train 81건(0.31%)  test 12건(0.14%)


C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2803841361.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_high_wind_caution"] = (df[f"{group_col}_ws_est"] >= HIGH_WIND_THRESHOLD).astype(int)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2803841361.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{group_col}_high_wind_caution"] = (df[f"{group_col}_ws_est"] >= HIGH_WIND_THRESHOLD).astype(int)
C:\Users\cho03\AppData\Local\Temp\ipykernel_25828\2803841361.py:5: PerformanceWarning: DataFrame is high

**확인할 것**: train·test 둘 다 0건은 아닌지(0건이면 아무 학습 신호가 없는 상수 컬럼이라는 뜻이라 의미가 없습니다). 건수가 매우 적은 그룹(특히 group_1)은 `04_model_selection`에서 이 피처가 실제로 쓸모 있는지(feature importance 등으로) 반드시 확인해야 합니다 — 표본이 적어 우연히 낀 노이즈일 수도 있습니다.

## 14. 저장

**사용 금지 컬럼 안내(중요)**: `train_base`에는 `scada_ws_kpx_group_*`(3-1에서 만듦), `scada_kpx_group_*`(01_preprocessing에서 만듦), `year`(3-2 검증용 임시 컬럼) 이렇게 SCADA 기반·분석용 컬럼이 섞여 있습니다. **이 컬럼들은 test_base에 없으므로 실제 모델 입력 피처로 쓰면 안 됩니다** — `04_model_selection.ipynb`에서 피처 목록을 고를 때 반드시 제외해야 합니다. 저장 자체는 그대로 두어(나중에 분석·검증용으로 다시 쓸 수 있게) 이 노트북에서 지우지는 않습니다.

In [30]:
print("train_features_v1:", train_base.shape)
print("test_features_v1:", test_base.shape)

train_base.to_parquet(f"{PROCESSED_DIR}/train_features_v1.parquet", index=False)
test_base.to_parquet(f"{PROCESSED_DIR}/test_features_v1.parquet", index=False)

print("저장 완료")

train_features_v1: (26304, 888)
test_features_v1: (8760, 860)
저장 완료


**확인할 것**: `data/processed/train_features_v1.parquet`, `test_features_v1.parquet` 파일이 생겼는지.

## 요약 및 다음 단계

민석님이 이중 관점(DS·도메인 전문가)으로 재검토를 요청해서 지적된 5가지를 전부 반영했습니다:

1. **회귀계수 불안정(다중공선성) → Ridge로 해결(3-2c)**: `RIDGE_ALPHA=30`, out-of-sample 성능 유지하면서 계수 크기 안정화. 3-3/3-5/12-2/12-3 전부 Ridge 사용.
2. **공기밀도 보정 효과 검증(4-1)**: 실제로는 거의 차이 없음(±0.001 수준)을 솔직하게 확인·기록. 그래도 도메인 근거가 있어 후보로는 유지.
3. **파워커브 변환 피처(12절, 최우선 과제였음)**: `{group}_power_curve_est` — SCADA (풍속,발전량) 곡선에 `ws_est`를 통과시킨 발전량 프록시. `ws_est`보다 뚜렷이 개선(out-of-sample로도 확인), fold-safe 버전까지 처음부터 같이 만듦(`_cv_*`).
4. **파워커브 구간 더미(13-1)**: calm/ramp/rated 3구간(컷아웃 구간은 `ws_est` 분포상 의미가 없어서 제외 — 이유를 셀에 그대로 남김).
5. **고풍속 주의 피처(13-2)**: 원래 "컷아웃 근접(25m/s)"으로 하려 했으나 train/test 모두 0건이라 상수 컬럼이 됨 → 실제로 신호가 있는 15m/s 기준의 "고풍속 주의"로 정직하게 바꿈.

**새로 생긴 컬럼**: `{group}_power_curve_est`(+ `_cv_2023_07/2024_01/2024_07`), `{group}_regime_calm/ramp/rated`, `{group}_high_wind_caution`.

**`04_model_selection.ipynb`에서 반드시 지킬 것(갱신됨)**:
- A안(2024 검증) → `ws_est_cv_2024_01`, `power_curve_est_cv_2024_01`
- B안 fold1(2023.07~12) → `ws_est_cv_2023_07`, `power_curve_est_cv_2023_07`
- B안 fold2(2024.01~06) → `ws_est_cv_2024_01`, `power_curve_est_cv_2024_01`
- B안 fold3(2024.07~12) → `ws_est_cv_2024_07`, `power_curve_est_cv_2024_07`
- 최종 전체 재학습(2025년 예측용) → `ws_est`, `power_curve_est`(전체 데이터 학습판)
- 모델 입력 피처 목록에서 `scada_ws_kpx_group_*`, `scada_kpx_group_*`, `year` 반드시 제외
- `{group}_high_wind_caution`(group_1은 표본이 매우 적음)이 실제로 쓸모 있는지 feature importance로 확인

**실행 방법**: 3-2c부터 순서대로 실행해 주세요(Ridge로 바뀐 3-3/3-5도 다시 실행되어야 합니다). 12절(파워커브 변환)과 13절(구간 더미) 결과를 확인해 주시면 `reports/03_features.md`·`HANDOFF.md`에 반영하고 `04_model_selection.ipynb`로 넘어가겠습니다.